# Attention Pooling Only - Accent Classification
Simple model using only attention pooling to aggregate temporal features and get a classification score.

In [ ]:
!pip install torch scikit-learn pandas tqdm

In [ ]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from sklearn.metrics import accuracy_score, f1_score
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# Mount Google Drive (Colab)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    COLAB = True
except ImportError:
    COLAB = False
    print("Not in Colab")

if COLAB:
    ROOT = "/content/drive/MyDrive/Hackenza_MUPlovers"
    FEATURE_DIR = os.path.join(ROOT, "cache/features_normalized")
    TRAIN_MANIFEST = os.path.join(ROOT, "data/train_manifest_split.csv")
    VAL_MANIFEST = os.path.join(ROOT, "data/val_manifest_split.csv")
else:
    # Local fallback paths
    FEATURE_DIR = "./features_normalized"
    TRAIN_MANIFEST = "./train_manifest_split.csv"
    VAL_MANIFEST = "./val_manifest_split.csv"

print(f"Feature dir: {FEATURE_DIR}")
print(f"Feature dir exists: {os.path.exists(FEATURE_DIR)}")

## Dataset

In [ ]:
class NativeDataset(Dataset):
    def __init__(self, manifest_csv, feature_dir):
        self.df = pd.read_csv(manifest_csv)
        self.feature_dir = feature_dir

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        file_id = row['file_id']
        label = int(row['label'])  # 0=non-native, 1=native
        
        # Load features: shape (T, 783)
        feature_path = os.path.join(self.feature_dir, f"{file_id}.npy")
        features = np.load(feature_path).astype(np.float32)
        features_tensor = torch.from_numpy(features)
        
        label_tensor = torch.tensor(label, dtype=torch.long)
        
        return features_tensor, label_tensor

In [ ]:
def collate_fn(batch):
    """Pad sequences and create attention mask."""
    sequences, labels = zip(*batch)
    
    # Get lengths before padding
    lengths = [seq.shape[0] for seq in sequences]
    
    # Pad sequences: (B, T_max, 783)
    padded = pad_sequence(sequences, batch_first=True, padding_value=0.0)
    
    # Create mask: 1 for real positions, 0 for padding
    mask = torch.zeros(padded.shape[:2])
    for i, length in enumerate(lengths):
        mask[i, :length] = 1.0
    
    labels_tensor = torch.stack(labels)
    
    return padded, mask, labels_tensor

In [ ]:
# Load datasets
if os.path.exists(TRAIN_MANIFEST) and os.path.exists(FEATURE_DIR):
    train_dataset = NativeDataset(TRAIN_MANIFEST, FEATURE_DIR)
    val_dataset = NativeDataset(VAL_MANIFEST, FEATURE_DIR)
    
    train_loader = DataLoader(
        train_dataset,
        batch_size=8,
        shuffle=True,
        collate_fn=collate_fn
    )
    
    val_loader = DataLoader(
        val_dataset,
        batch_size=8,
        shuffle=False,
        collate_fn=collate_fn
    )
    
    print(f"Train samples: {len(train_dataset)}")
    print(f"Val samples: {len(val_dataset)}")
    DATA_READY = True
else:
    print("Manifest or feature dir not found. Will create dummy data.")
    DATA_READY = False

## Model Architecture

In [ ]:
class AttentionPoolingModel(nn.Module):
    """
    Simple model: Attention pooling only.
    
    Input: (B, T, 783) features
    1. Apply learned attention weights to each time step
    2. Weighted sum to get (B, 783) pooled representation
    3. Classification head: (B, 783) -> (B, 2) logits
    """
    def __init__(self, feature_dim=783, hidden_dim=128):
        super().__init__()
        
        # Attention: compute scalar score for each time step
        self.attention = nn.Linear(feature_dim, 1)
        
        # Classification head
        self.classifier = nn.Sequential(
            nn.Linear(feature_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, 2)  # Binary classification
        )
    
    def forward(self, x, mask=None):
        """
        Args:
            x: (B, T, 783) features
            mask: (B, T) binary mask (1 for real, 0 for padding)
        
        Returns:
            logits: (B, 2) classification scores
        """
        # Compute attention scores: (B, T, 1)
        attn_scores = self.attention(x)
        
        # Apply mask (optional)
        if mask is not None:
            # Set padding positions to large negative value
            mask_expanded = mask.unsqueeze(-1)  # (B, T, 1)
            attn_scores = attn_scores.masked_fill(mask_expanded == 0, -1e9)
        
        # Softmax over time dimension: (B, T, 1)
        attn_weights = F.softmax(attn_scores, dim=1)
        
        # Weighted sum over time: (B, 783)
        pooled = (x * attn_weights).sum(dim=1)
        
        # Classification
        logits = self.classifier(pooled)  # (B, 2)
        
        return logits

In [ ]:
# Create model
model = AttentionPoolingModel(feature_dim=783, hidden_dim=128).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

print(f"Model created on {device}")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")

## Training

In [ ]:
def train_epoch():
    """Train for one epoch."""
    model.train()
    total_loss = 0.0
    
    for x, mask, y in tqdm(train_loader, desc="Training"):
        x = x.to(device)
        mask = mask.to(device)
        y = y.to(device)
        
        optimizer.zero_grad()
        logits = model(x, mask)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    return total_loss / len(train_loader)

In [ ]:
def evaluate():
    """Evaluate on validation set."""
    model.eval()
    all_preds = []
    all_labels = []
    total_loss = 0.0
    
    with torch.no_grad():
        for x, mask, y in tqdm(val_loader, desc="Evaluating"):
            x = x.to(device)
            mask = mask.to(device)
            y = y.to(device)
            
            logits = model(x, mask)
            loss = criterion(logits, y)
            total_loss += loss.item()
            
            preds = logits.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y.cpu().numpy())
    
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average='weighted', zero_division=0)
    avg_loss = total_loss / len(val_loader)
    
    return acc, f1, avg_loss

In [ ]:
if DATA_READY:
    EPOCHS = 10
    best_f1 = 0.0
    
    for epoch in range(EPOCHS):
        train_loss = train_epoch()
        val_acc, val_f1, val_loss = evaluate()
        
        print(f"\nEpoch {epoch+1}/{EPOCHS}")
        print(f"  Train Loss: {train_loss:.4f}")
        print(f"  Val Loss:   {val_loss:.4f}")
        print(f"  Val Acc:    {val_acc:.4f}")
        print(f"  Val F1:     {val_f1:.4f}")
        
        if val_f1 > best_f1:
            best_f1 = val_f1
            torch.save(model.state_dict(), "best_attention_model.pth")
            print("  ✓ Saved best model")
else:
    print("Skipping training: Data not available. Create dummy data or provide manifest files.")

## Inference

In [ ]:
def get_predictions(x, mask=None):
    """
    Get attention-pooled scores for input features.
    
    Args:
        x: (B, T, 783) features
        mask: (B, T) mask
    
    Returns:
        probs: (B, 2) probabilities [non-native, native]
        preds: (B,) predicted class (0 or 1)
    """
    model.eval()
    with torch.no_grad():
        logits = model(x, mask)
        probs = F.softmax(logits, dim=1)
        preds = logits.argmax(dim=1)
    return probs, preds


# Example usage (dummy data)
print("\n=== Example Inference ===")
B, T, D = 2, 50, 783  # Batch size, max time, feature dim
test_x = torch.randn(B, T, D).to(device)
test_mask = torch.ones(B, T).to(device)
test_mask[0, 40:] = 0  # Mask padding in first sample

probs, preds = get_predictions(test_x, test_mask)
print(f"Input shape: {test_x.shape}")
print(f"Probabilities shape: {probs.shape}")
print(f"Predictions: {preds.cpu().numpy()}")
print(f"\nSample 0 - Non-native: {probs[0, 0]:.4f}, Native: {probs[0, 1]:.4f}")
print(f"Sample 1 - Non-native: {probs[1, 0]:.4f}, Native: {probs[1, 1]:.4f}")